# End-to-End Churn Prediction Model Using AWS SageMaker

This notebook is structured to match the assignment workflow:

1. Confirm environment and configuration  
2. Upload or reference the raw CSV converted from the Kaggle XLSX  
3. Build and run a **SageMaker Pipeline** with:
   - preprocessing
   - XGBoost training
   - AUC evaluation
   - conditional model registration  
4. Create and ingest a **SageMaker Feature Store** feature group  
5. Query the Feature Store offline and online stores  
6. Approve and deploy the model  
7. Test the endpoint  
8. Clean up resources

The project uses the SageMaker built-in **XGBoost** algorithm with `binary:logistic` and evaluates with **AUC-ROC**.

## 0. Notes before running

- Convert the Kaggle `.xlsx` dataset to `storedata.csv`.
- Upload this project folder to SageMaker Studio.
- Update `config/project_config.json` before running.
- The preprocessing step assumes the target column is named **`retained`**.
- The built-in XGBoost container expects the label in the **first column** and no headers.

In [ ]:
!pip install -q -r ../requirements.txt

In [ ]:
import json
import time
from pathlib import Path

import boto3
import pandas as pd
import sagemaker
from sagemaker import get_execution_role

BASE_DIR = Path("..").resolve()
CONFIG_PATH = BASE_DIR / "config" / "project_config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

cfg

## 1. Set up session

In [ ]:
boto_sess = boto3.Session(region_name=cfg["region"])
sm_client = boto_sess.client("sagemaker")
session = sagemaker.Session(boto_session=boto_sess)
role = get_execution_role()

print("Region:", cfg["region"])
print("Role:", role)
print("Default bucket:", session.default_bucket())

## 2. Review the pipeline code

In [ ]:
print((BASE_DIR / "pipeline.py").read_text()[:3000])

## 3. Create or update and start the pipeline

In [ ]:
%run ../pipeline.py

After running the previous cell, note the printed pipeline execution ARN. You can also inspect it directly below.

In [ ]:
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.pipeline import Pipeline

pipeline_session = PipelineSession(boto_session=boto_sess)
pipeline_name = cfg["pipeline_name"]
pipeline = Pipeline(name=pipeline_name, sagemaker_session=pipeline_session)

execution = pipeline.start(
    parameters=dict(
        InputDataS3Uri=cfg["input_data_s3_uri"],
        PipelineRoot=cfg["pipeline_root"],
        ProcessingInstanceType=cfg["processing_instance_type"],
        TrainingInstanceType=cfg["training_instance_type"],
        TrainingInstanceCount=cfg["training_instance_count"],
        AucThreshold=cfg["auc_threshold"],
        ModelPackageGroupName=cfg["model_package_group_name"],
        FeatureGroupName=cfg["feature_group_name_prefix"],
        FeatureStoreOfflineS3Uri=cfg["feature_store_offline_s3_uri"],
    )
)
print("Pipeline execution ARN:", execution.arn)

## 4. Wait for the pipeline to finish

In [ ]:
execution.describe()

In [ ]:
# Poll until the execution completes
while True:
    desc = execution.describe()
    status = desc["PipelineExecutionStatus"]
    print("Status:", status)
    if status in ["Succeeded", "Failed", "Stopped"]:
        break
    time.sleep(30)

## 5. Inspect pipeline steps

In [ ]:
steps = execution.list_steps()
steps

## 6. Review evaluation metrics

The pipeline writes `evaluation.json` as a processing output and uses it to decide whether the model should be registered.

In [ ]:
for step in steps:
    print(step["StepName"], step["StepStatus"])

## 7. Find preprocessing outputs from the pipeline execution

We need the preprocessing output S3 URI to create and ingest the Feature Store data file.

In [ ]:
pipeline_desc = execution.describe()
pipeline_desc

In [ ]:
# Locate the preprocessing job ARN from pipeline steps
preprocess_step = next(s for s in steps if s["StepName"] == "PreprocessChurnData")
preprocess_arn = preprocess_step["Metadata"]["ProcessingJob"]["Arn"]
preprocess_job_name = preprocess_arn.split("/")[-1]
print("Processing job:", preprocess_job_name)

processing_desc = sm_client.describe_processing_job(ProcessingJobName=preprocess_job_name)
processing_desc["ProcessingOutputConfig"]["Outputs"]

## 8. Create and ingest a Feature Group

This uses the `feature_store_ingest.csv` output from preprocessing.  
A unique feature group name is generated so repeated notebook runs do not collide.

In [ ]:
feature_store_output = next(
    o for o in processing_desc["ProcessingOutputConfig"]["Outputs"]
    if o["OutputName"] == "featurestore"
)
feature_store_ingest_s3_uri = feature_store_output["S3Output"]["S3Uri"].rstrip("/") + "/feature_store_ingest.csv"
feature_group_name = f"{cfg['feature_group_name_prefix']}-{int(time.time())}"

print("Feature store ingest file:", feature_store_ingest_s3_uri)
print("Feature group name:", feature_group_name)

In [ ]:
!python ../create_and_ingest_feature_group.py   --feature-group-name {feature_group_name}   --input-csv {feature_store_ingest_s3_uri}

## 9. Query the Feature Store offline store

This runs an Athena query to retrieve the latest non-deleted record per `record_id`.

In [ ]:
athena_output = f"s3://{session.default_bucket()}/athena-query-results/"
print(athena_output)

In [ ]:
!python ../query_feature_store.py   --feature-group-name {feature_group_name}   --mode online   --record-ids {" ".join(sample_record_ids)}

## 10. Review model packages in the registry

If the AUC is above the threshold, the model is registered with approval status `PendingManualApproval`.

In [ ]:
model_packages = sm_client.list_model_packages(
    ModelPackageGroupName=cfg["model_package_group_name"],
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=5,
)
model_packages["ModelPackageSummaryList"]

## 11. Approve the latest model package

Run this cell only after you verify the evaluation metrics.

In [ ]:
latest_package_arn = model_packages["ModelPackageSummaryList"][0]["ModelPackageArn"]
sm_client.update_model_package(
    ModelPackageArn=latest_package_arn,
    ModelApprovalStatus="Approved"
)
print("Approved:", latest_package_arn)

## 12. Deploy the approved model from the registry

In [ ]:
endpoint_name = f"tea-store-churn-xgb-{int(time.time())}"
print(endpoint_name)

In [ ]:
!python ../deploy_from_registry.py   --approval-status Approved   --endpoint-name {endpoint_name}

## 13. Invoke the endpoint with a sample row

The built-in XGBoost endpoint expects CSV values only, without the target column.

In [ ]:
runtime = boto_sess.client("sagemaker-runtime")

# Pull one feature row from the feature store ingest file for a quick test
sample_df = pd.read_csv(feature_store_ingest_s3_uri)
feature_cols = [c for c in sample_df.columns if c not in ["record_id", "event_time", "retained"]]
payload = ",".join(map(str, sample_df.loc[0, feature_cols].tolist()))

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Body=payload,
)
print(response["Body"].read().decode("utf-8"))

In [ ]:
predictor_sm = sagemaker.Predictor(endpoint_name=endpoint_name, sagemaker_session=session)
predictor_sm.delete_endpoint(delete_endpoint_config=True)
print("Deleted endpoint and endpoint config:", endpoint_name)